# Production Simulation on Kaggle

Dua mode, pilih lewat `SOURCE_MODE`:

1. **`live`** - rekam HLS 30 detik -> proses -> replay, diulang `LIVE_CYCLES` kali.
   Ini mode default.
2. **`drive_video`** - unduh satu video 1:02:26 dari Google Drive, proses per 30 detik,
   lalu replay tiap window ke pipeline.

**Alur:** cek lingkungan -> clone/install -> set secret -> jalankan mode -> unduh bundle.

> Kaggle: Settings -> Accelerator -> **GPU T4 x2** (opsional), Internet **ON**.
> Default pipeline adalah **single pass**. Untuk anotasi satu video dengan 2xT4,
> gunakan `notebooks/colab_annotate_tmp_kaggle.ipynb`.

In [ ]:
import shutil

import torch

print("torch      :", torch.__version__)
print("CUDA ready :", torch.cuda.is_available())
print("GPU count  :", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}    : {props.name} ({props.total_memory / 1e9:.1f} GB)")

print("ffmpeg     :", shutil.which("ffmpeg"))
print("ffprobe    :", shutil.which("ffprobe"))

from pathlib import Path

for folder in ("/kaggle/input", "/kaggle/working"):
    if Path(folder).exists():
        usage = shutil.disk_usage(folder)
        print(f"{folder}: {usage.free / 1e9:.1f} GB free")

## 1. Clone dan install

Kaggle sudah membawa PyTorch CUDA dan ffmpeg; jangan install wheel torch CPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/ibrahim119920/highway-lane-analytics.git"
PROJECT = "highway-lane-occupancy"

if os.path.basename(os.getcwd()) != PROJECT:
    if not Path(PROJECT).is_dir():
        subprocess.run(["git", "clone", REPO, PROJECT], check=True)
    os.chdir(PROJECT)

ROOT = Path.cwd()
SRC = ROOT / "src"
assert (SRC / "lane_analytics").is_dir(), f"Repo tidak lengkap: {SRC}"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"],
    check=True,
)

import ultralytics
from lane_analytics.colab import activate
from lane_analytics.pipeline import process_video
from lane_analytics.replay import replay_run

print("repo         :", ROOT)
print("ultralytics  :", ultralytics.__version__)
print("activate     :", activate.__module__)

## 2. Pilih mode dan parameter

`SOURCE_MODE = "live"` menjalankan opsi 1 (HLS). `SOURCE_MODE = "drive_video"`
menjalankan opsi 2 (video Drive 1:02:26 per 30 detik).

Untuk replay ke pipeline:

1. Tambahkan secret `WEBHOOK_URL` (wajib) dan `WEBHOOK_TOKEN` (opsional) di
   **Add-ons -> Secrets**.
2. Set `REPLAY = True`.
3. `DRY_RUN = True` untuk menulis batch ke `output/replay/` tanpa HTTP.
   `DRY_RUN = False` untuk benar-benar POST ke webhook.
4. `REPLAY = False` untuk melewati replay sepenuhnya.

> Catatan: `activate()` hanya memproses replay bila `replay=True`. Jadi untuk uji
> dry-run, gunakan `REPLAY = True` dan `DRY_RUN = True`.

In [ ]:
#@title Mode dan parameter { display-mode: "form" }
SOURCE_MODE = "live"  #@param ["live", "drive_video"]

# --- opsi 1: live HLS ------------------------------------------------------
LIVE_URL = "https://pub2.hk-opt.com/LiveApp/streams/501264524429471982550467.m3u8"
LIVE_CYCLES = 3            # jumlah siklus rekam -> proses -> replay
RECORD_SECONDS = 30        # durasi rekaman per siklus
CYCLE_SLEEP_SECONDS = 0.0  # jeda antar siklus (0 = langsung lanjut)

# --- opsi 2: satu video Drive 1:02:26 -------------------------------------
DRIVE_FILE_ID = "1mgunh3mQX8YYM4doPSNSSG1fITkQvvFd"
DRIVE_URL = f"https://drive.google.com/uc?id={DRIVE_FILE_ID}"
DRIVE_VIDEO_PATH = "/kaggle/working/source_62min.mp4"
WINDOW_SECONDS = 30        # olah per 30 detik
MAX_WINDOWS = 0            # 0 = semua (~124 window)
# GANTI dengan waktu overlay frame 0 video. Wajib untuk mode drive_video.
BASE_TIMESTAMP = "2026-11-08T00:00:00"

# --- model dan replay ------------------------------------------------------
FRAME_STRIDE = 1
IMGSZ = 960
REPLAY = False             # True = aktifkan replay
DRY_RUN = True             # True = tulis batch, tanpa HTTP
SPEED = 5.0                # 1 = real time, 5 = 5x
REPLAY_WINDOW_SECONDS = 5.0
# ---------------------------------------------------------------------------

assert SOURCE_MODE in {"live", "drive_video"}, SOURCE_MODE
print("mode          :", SOURCE_MODE)
print("replay        :", REPLAY, "dry_run:", DRY_RUN)
if SOURCE_MODE == "live":
    print("live url      :", LIVE_URL)
    print("cycles        :", LIVE_CYCLES, "x", RECORD_SECONDS, "s")
else:
    print("drive video   :", DRIVE_URL)
    print("window        :", WINDOW_SECONDS, "s, max:", MAX_WINDOWS or "all")
    print("base timestamp:", BASE_TIMESTAMP)

## 3. Secret dan config efektif

Config asli disalin ke `/kaggle/working/config_kaggle.yaml` dengan evidence dimatikan,
sehingga setiap siklus/window tidak membuat snapshot dan klip 8 detik.

In [ ]:
from pathlib import Path

import yaml

WEBHOOK_URL = ""
WEBHOOK_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    WEBHOOK_URL = _secrets.get_secret("WEBHOOK_URL")
    WEBHOOK_TOKEN = _secrets.get_secret("WEBHOOK_TOKEN")
except Exception as exc:
    print("Kaggle Secrets tidak tersedia:", type(exc).__name__, exc)

DO_REPLAY = bool(REPLAY and (DRY_RUN or WEBHOOK_URL))
if REPLAY and not DO_REPLAY:
    print("warning: WEBHOOK_URL kosong; replay HTTP dilewati. Set DRY_RUN=True untuk uji.")

CONFIG_PATH = Path("/kaggle/working/config_kaggle.yaml")
raw = yaml.safe_load(Path("config/config.yaml").read_text(encoding="utf-8"))
raw["evidence"]["enabled"] = False
raw["video"]["frame_stride"] = FRAME_STRIDE
raw["model"]["imgsz"] = IMGSZ
CONFIG_PATH.write_text(yaml.safe_dump(raw, sort_keys=False), encoding="utf-8")

print("webhook       :", "set" if WEBHOOK_URL else "empty")
print("do replay     :", DO_REPLAY)
print("config        :", CONFIG_PATH)

## 4. Opsi 1 - live HLS

Setiap siklus:

1. `activate(source="live")` merekam `RECORD_SECONDS` detik dari `LIVE_URL`.
2. Pipeline single-pass mendeteksi kendaraan, lane events, dan hogging.
3. Events/tracks di-replay ke webhook (atau dry-run).
4. Siklus diulang `LIVE_CYCLES` kali.

Jalankan cell ini hanya jika `SOURCE_MODE = "live"`.

In [ ]:
import time

live_results = []

if SOURCE_MODE == "live":
    device = "0" if torch.cuda.is_available() else "cpu"
    for cycle in range(1, int(LIVE_CYCLES) + 1):
        print(f"\n=== LIVE CYCLE {cycle}/{LIVE_CYCLES} ===")
        started = time.time()
        try:
            result = activate(
                source="live",
                stream_url=LIVE_URL,
                record_seconds=RECORD_SECONDS,
                video_dir="data/raw/live",
                config_path=CONFIG_PATH,
                device=device,
                imgsz=IMGSZ,
                frame_stride=FRAME_STRIDE,
                write_video=False,
                replay=DO_REPLAY,
                webhook_url=WEBHOOK_URL or None,
                webhook_token=WEBHOOK_TOKEN or None,
                speed=SPEED,
                window_seconds=REPLAY_WINDOW_SECONDS,
                dry_run=DRY_RUN,
            )
            live_results.append(result)
            print(result)
            for item in result.videos:
                print(
                    f"  {item['video']} | status={item['status']} | "
                    f"events={item.get('events', '-')} | vehicles={item.get('vehicles', '-')}"
                )
                if item.get("error"):
                    print("  error:", item["error"])
        except Exception as exc:
            print("cycle failed:", type(exc).__name__, exc)
            live_results.append({"status": "error", "error": str(exc)})

        wall = time.time() - started
        print(f"cycle wall: {wall:.1f}s")
        if CYCLE_SLEEP_SECONDS > 0:
            time.sleep(CYCLE_SLEEP_SECONDS)
else:
    print("SOURCE_MODE bukan 'live'; lewati cell ini.")

## 5. Opsi 2 - satu video Drive 1:02:26, per 30 detik

Video diunduh sekali, lalu diproses dalam window 30 detik. Setiap window:

1. `process_video(start_frame=..., frame_limit=...)` - hanya 30 detik sumber.
2. Events/tracks ditulis ke `output/`.
3. Replay window tersebut ke webhook (atau dry-run).
4. Jika proses + replay selesai sebelum 30 detik, notebook menunggu sisanya agar
   ritme mendekati real-time.

Catatan: tracker dan track ID di-reset tiap window. Pipeline penerima harus memakai
`run_id` per window dan tidak mengharapkan ID kontinu antar window. Jalankan cell ini
hanya jika `SOURCE_MODE = "drive_video"`.

In [ ]:
import math
import time
from pathlib import Path

import cv2
import gdown

from lane_analytics.config import load_config

drive_runs = []

if SOURCE_MODE == "drive_video":
    if not BASE_TIMESTAMP:
        raise ValueError("BASE_TIMESTAMP wajib diisi untuk mode drive_video")

    video_path = Path(DRIVE_VIDEO_PATH)
    if not video_path.exists():
        gdown.download(DRIVE_URL, str(video_path), quiet=False, fuzzy=True)
    if not video_path.exists() or video_path.stat().st_size == 0:
        raise FileNotFoundError(f"Download gagal: {video_path}")

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Video tidak bisa dibuka: {video_path}")
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    capture.release()
    if fps <= 0 or total_frames <= 0:
        raise RuntimeError("FPS/frame count tidak terbaca")

    window_frames = max(1, int(round(WINDOW_SECONDS * fps)))
    total_windows = int(math.ceil(total_frames / window_frames))
    if MAX_WINDOWS:
        total_windows = min(total_windows, int(MAX_WINDOWS))

    print("video        :", video_path, f"({video_path.stat().st_size / 1e6:.1f} MB)")
    print("fps/frames   :", round(fps, 3), "/", total_frames)
    print("window frames:", window_frames)
    print("windows      :", total_windows)

    device = "0" if torch.cuda.is_available() else "cpu"
    for index in range(total_windows):
        start = index * window_frames
        limit = min(window_frames, total_frames - start)
        run_id = f"{video_path.stem}_w{index:04d}"
        print(f"\n=== WINDOW {index + 1}/{total_windows} | frames {start}-{start + limit} ===")

        config = load_config(CONFIG_PATH)
        config.video.path = str(video_path)
        config.video.frame_stride = FRAME_STRIDE
        config.video.max_frames = None
        config.base_timestamp = BASE_TIMESTAMP
        config.model.device = device
        config.model.imgsz = IMGSZ
        config.evidence.enabled = False

        started = time.time()
        batch = process_video(
            config,
            start_frame=start,
            frame_limit=limit,
            write_video=False,
            generate_reports=False,
            run_id=run_id,
        )
        replay_info = None
        if DO_REPLAY:
            replay_info = replay_run(
                batch.events_path,
                tracks_path=batch.tracks_path,
                url=WEBHOOK_URL or None,
                token=WEBHOOK_TOKEN or None,
                run_id=run_id,
                speed=SPEED,
                window_seconds=REPLAY_WINDOW_SECONDS,
                dry_run=DRY_RUN,
                dry_run_dir="output/replay",
            ).as_dict()

        wall = time.time() - started
        item = {
            "window": index,
            "start_frame": start,
            "frames": limit,
            "run_id": run_id,
            "events": batch.total_events,
            "vehicles": batch.total_vehicles,
            "hogging": batch.hogging_events,
            "wall_seconds": round(wall, 2),
            "replay": replay_info,
        }
        drive_runs.append(item)
        print(
            f"events={item['events']} vehicles={item['vehicles']} "
            f"hogging={item['hogging']} wall={wall:.1f}s"
        )

        # Jaga ritme 30 detik bila proses + replay lebih cepat.
        if wall < WINDOW_SECONDS:
            time.sleep(WINDOW_SECONDS - wall)
        else:
            print(f"behind schedule by {wall - WINDOW_SECONDS:.1f}s")
else:
    print("SOURCE_MODE bukan 'drive_video'; lewati cell ini.")

## 6. Package hasil

Bundle berisi `manifest.json` + seluruh `output/` (events, tracks, replay dry-run).
Tidak ada video anotasi di mode production ini agar hemat disk dan waktu.
Unduh dari tab **Output** Kaggle sebelum sesi berakhir.

In [ ]:
import json
import zipfile
from pathlib import Path

if SOURCE_MODE == "live":
    runs = [
        item.as_dict() if hasattr(item, "as_dict") else item
        for item in live_results
    ]
else:
    runs = drive_runs

manifest = {
    "source_mode": SOURCE_MODE,
    "settings": {
        "frame_stride": FRAME_STRIDE,
        "imgsz": IMGSZ,
        "replay": REPLAY,
        "dry_run": DRY_RUN,
        "speed": SPEED,
        "live_url": LIVE_URL if SOURCE_MODE == "live" else None,
        "live_cycles": LIVE_CYCLES if SOURCE_MODE == "live" else None,
        "record_seconds": RECORD_SECONDS if SOURCE_MODE == "live" else None,
        "drive_file_id": DRIVE_FILE_ID if SOURCE_MODE == "drive_video" else None,
        "window_seconds": WINDOW_SECONDS if SOURCE_MODE == "drive_video" else None,
        "base_timestamp": BASE_TIMESTAMP if SOURCE_MODE == "drive_video" else None,
    },
    "runs": runs,
}

out = Path("output")
work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
manifest_path = work_dir / "production_manifest.json"
bundle = work_dir / "production_bundle.zip"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(manifest_path, "manifest.json")
    for path in sorted(out.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(out.parent))

print(f"Manifest : {manifest_path}")
print(f"Bundle   : {bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
print(f"Runs     : {len(runs)}")

## 7. Catatan production

- **Live stream** membutuhkan ffmpeg/ffprobe dan URL HLS yang bisa diakses dari
  Kaggle. Notebook merekam 30 detik per siklus; jika pemrosesan lebih lama dari 30
  detik, siklus berikutnya otomatis berjalan setelah selesai (tidak ada frame yang
  tumpang tindih).
- **Video Drive 1:02:26** diproses per 30 detik. `BASE_TIMESTAMP` wajib diisi dari
  overlay video; jika tidak, timestamp event akan bergeser.
- **Track ID** di-reset per window/siklus. Pipeline penerima harus memakai `run_id`
  per window dan melakukan dedupe berdasarkan `event_id`/`run_id`.
- **Replay at-least-once.** Receiver wajib dedupe; pengiriman bisa retry.
- **Kaggle bukan scheduler.** Tidak ada cron native. Untuk production berulang,
  picu notebook lewat Kaggle API/CI, atau pindahkan job ke VM/container dengan cron.
- **Batas sesi/kuota.** Gunakan **Save & Run All** dan unduh bundle sebelum sesi
  berakhir. Untuk 2xT4, gunakan notebook anotasi dengan `WORKERS = 2`,
  `DEVICES = "0,1"`.
- **Kontrak data.** Events/tracks mengikuti `docs/schema/lane_event.schema.json` dan
  `docs/HANDOFF_DATA_PIPELINE_DASHBOARD.md`.